# Data Cleaning — Construction Spec (principles, not a recipe)

**Purpose.** This notebook is the **source-of-truth spec** for data cleaning, the one-time pass that turns the raw Sharadar CSVs into the cleaned parquet layer under `data/cleaned/` that every later step reads instead of re-parsing CSVs. It is deliberately unlike the factor specs: it is **not** a stage-by-stage recipe. Cleaning is a craft, and the right choices depend on your data vintage and how far you take the model — so this spec gives the **intuition and the general data-hygiene principles** that make a foundation you can trust, and leaves the implementation to you.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. There are no numbered stages to follow here; read the principles, then design the cleaning your data actually needs.

**Deliverable.** A cleaned, typed, point-in-time-safe parquet layer under `data/cleaned/` — one folder per table you consume, partitioned and compressed. See `data/README.md` for the raw schemas and field-level notes; this spec is about *what makes the result trustworthy*, not which columns exist.

**Scope.** Clean what you will consume. The factor pipeline reads a handful of tables — SF1 (fundamentals), SEP (prices), DAILY (market caps), TICKERS (metadata), ACTIONS (corporate actions); the rest of the Sharadar bundle is useful, but not consumed in this course. And keep the boundary clean: this step is **mechanical hygiene** — typing, units, point-in-time keys, dedup, partitioning. **Universe selection is not cleaning** — deciding which stocks are investable is the ESTU's job (step 01). Don't fold an investability filter in here.

## Why this step is load-bearing

Everything downstream inherits your cleaning. A factor is just a transformation of these tables; a wrong unit, a look-ahead join, or a silent duplicate does not produce one bug — it biases every factor that touches the affected column, in a *correlated* way that is hard to spot later because the numbers still look plausible. The cheapest place to catch a data error is here, before it has been laundered through a standardization and a regression.

So the goal is not a pristine dataset — vendor data is never pristine. The goal is a **trustworthy, reproducible** one: a layer where every later notebook can *assume*, without re-checking, that dates are dates, units are consistent, the time key is point-in-time, and there is exactly one row per entity-date. Cleaning is the contract that lets the rest of the repo stop worrying about the raw files.

> **Intuition.** Treat the raw CSVs as immutable evidence and `data/cleaned/` as the single canonical interpretation of them. Raw goes in, cleaned comes out, and nothing downstream ever reaches back to the raw layer. If a cleaning rule is wrong, you fix it in one place and rebuild — not in fifteen factor notebooks.

## What "clean" means here

Six properties define a trustworthy layer. None of them prescribe *how* — they describe the result you are checking for.

1. **Typed.** Dates parse to `Date` (not string, and not `Datetime` — you never want a time-of-day sneaking into a daily join). Numeric columns are numeric. Identifier columns are a stable integer type. Vendor CSVs are all strings until you say otherwise.
2. **Point-in-time.** Each table carries a time key that is the date the information was *available*, not the date it *describes*. For fundamentals that is the filing date, not the fiscal period end. This one property prevents the most damaging class of bug in the whole project.
3. **Unit-consistent.** Every column is in one known unit, fixed once. The classic trap: Sharadar market cap is in *millions* of dollars.
4. **De-duplicated.** Exactly one row per natural key (one price per ticker-date, one filing per ticker-period, and so on). Duplicates silently double-count in sums and pick an arbitrary value in joins.
5. **Survivorship-complete.** Delisted, dead, and merged-away names are *kept*. A dataset that contains only companies that still exist today is a time machine that makes every backtest look brilliant.
6. **Cheap to read.** Partitioned, compressed parquet (partition on the time key; zstd). A full-history CSV scan that takes seconds becomes a partitioned read that takes milliseconds — and you will read these tables hundreds of times.

If your cleaned layer has these six properties, the specific transformations that got you there don't much matter.

## A loose master list: the fields we pull, and where they live

This is **not** an exhaustive account of what Sharadar offers — the bundle carries far more than this model touches (institutional holdings, insider trades, 8-K events, ETF prices, and dozens of ratio columns you may well find uses for). It is the *working set*: the fields the factor pipeline actually reaches for, grouped by the file they come from, so you know what has to survive cleaning. Treat it as a floor, not a ceiling — clean these well, and pull more as a factor demands it. Field-level definitions and units live in `data/README.md`.

**TICKERS** — the entity master and metadata (one row per ticker; a current snapshot).
- `permaticker` — the stable entity id; your join and history key everywhere.
- `ticker` — the current symbol; a display label, not an identity.
- `category`, `exchange`, `currency` — eligibility metadata the ESTU screens on (step 01).
- `isdelisted` — for survivorship: you *keep* delisted names, so this is a flag to respect, not to filter on.
- `industry`, `sector` — the raw classification atoms the industry scheme (step 03) is built from; `famaindustry`, `sicsector`, `siccode` are coarser alternatives.
- `firstpricedate`, `lastpricedate` — the real listing window (a name's true entry and exit).

**SEP** — daily split-adjusted prices; your returns, momentum, volatility, and liquidity source.
- `ticker`, `date` — keys.
- `closeadj` — split- **and** dividend-adjusted close; the total-return series (Beta, Momentum, Residual Volatility).
- `closeunadj` — the raw traded close; for reconstructing shares outstanding and as the Dividend Yield price denominator.
- `volume` — traded share volume, for share turnover (Liquidity) and the ESTU's dollar-volume screen.
- (`close` is split-adjusted only — fine for charts, wrong for returns; `open`/`high`/`low` go unused here.)

**DAILY** — the daily fundamental snapshot; market cap on every calendar day.
- `ticker`, `date` — keys.
- `marketcap` — daily total market cap: the ESTU size screen, the Size factor, the Country regression weight, and the cap-weighted standardization mean every style uses. **It is in millions — normalize once.** (`ev`, `pb`, `pe`, `ps` ride along if a factor ever wants them.)

**SF1** — as-reported fundamentals; use the **ARQ** dimension and key point-in-time on `datekey`.
- `ticker`, `dimension`, `datekey`, `calendardate` — keys and the time columns (`datekey` = availability date; `calendardate` = period end — never key on the latter).
- `equity` — book equity (Book-to-Price, Leverage).
- `netinc`, `depamor` — net income and depreciation/amortization → cash earnings (Earnings Yield).
- `dps` — dividends per share (Dividend Yield).
- `debtnc`, `assets` — long-term debt and total assets (Leverage).
- `eps`, `sps` — EPS and sales-per-share, over five-year annual histories (Growth).
- `sharesbas` — shares outstanding from the filing cover page (a cross-check on the daily share count Liquidity imputes from `marketcap`/`closeunadj`).

**ACTIONS** — the corporate-actions log.
- `date`, `action`, `ticker`, `value` — the split and dividend events; used to rebase per-share dividends across splits for Dividend Yield.

**One non-Sharadar input.** The risk-free rate for excess returns (Beta, Momentum, Residual Volatility) is **not** in Sharadar — it comes from the Fama–French daily file (Ken French's data library, the `RF` column). It is the model's one external dependency; `data/README.md` covers its format.

## Principles and general data-cleaning tips

These are the ideas worth internalizing — most of them transfer to any vendor panel data, not just Sharadar.

**1. Point-in-time is the one you cannot get wrong.** A fundamental value must be keyed by *when you could have known it*. Sharadar gives you both a period-end date (`calendardate` / `reportperiod`) and a filing date (`datekey`); the filing lands 30–90 days after the period, and using the period end as your time key leaks the future into every fundamentals-based factor. **SUGGESTION:** time-key (and partition) fundamentals on the availability date. Then "what did I know as of date *t*?" is always `available_date <= t`, most-recent row per entity. Build this in at the cleaning layer so no downstream notebook can get it wrong.

**2. Keep the dead.** Do not filter out delisted names during cleaning. Survivorship bias is the most seductive error in quant research because it always flatters your results. Retain everything and let *time* remove a name — a stock leaves your universe when it stops having prices, not when a static `isdelisted` flag says so today.

**3. Normalize units once, at the boundary.** A value in the wrong unit is a factor-of-1000 error that silently corrupts every cutoff and every weight downstream. Fix units at ingestion — convert market cap from millions to dollars once, here — so no factor notebook ever has to remember to. A unit decision made in one place is a unit decision you cannot get inconsistently wrong.

**4. Null, zero, and missing are three different things.** A blank cell can mean "genuinely zero" (a firm with no long-term debt), "not reported this period," or "the vendor doesn't carry this field." They need different handling, and blindly filling nulls with zero — or dropping every null row — will silently delete or distort a real economic cohort. Decide per column, from the field's meaning, and write the decision down. When in doubt, preserve the distinction (keep the null) and let the consuming factor decide; it has the context.

**5. De-duplicate with a principled tiebreaker.** Pick the natural key, then decide *which* row wins when the key repeats. Usually you want the freshest: sort by a last-updated stamp and keep the latest per key, so restated rows replace stale ones. ⚠️ A subtlety that has bitten production systems: many "keep unique" operations do **not** promise to keep the first row of your sort order unless you explicitly ask them to — so you can sort correctly and still keep an arbitrary duplicate. Verify your dedup keeps the row you think it does, and spot-check a key you know has duplicates.

**6. Don't trust vendor type inference.** CSV readers guess types from a sample, and the guess flips when nulls appear — an integer identifier column with a few blanks gets read as float, and now your join keys don't match. Force the schema on the columns that matter (keys, dates, the numerics you will compute with) rather than trusting inference; cast date-like columns down to `Date`; coerce identifier columns to a stable integer type. Let the vendor guess only on columns you don't depend on.

**7. Use a stable entity key.** Tickers get recycled — a delisted `FOO` can become a different company's `FOO` years later. Chaining history on the ticker string splices two companies together at the reuse boundary. Key on the permanent entity id (`permaticker`) and treat the ticker as a display label that changes over time.

**8. Make the pass idempotent and safe to re-run.** You will run this more than once. A re-run on the same inputs should produce the same outputs and never a corrupt half-state: write to a temporary file and rename it into place, so a crash mid-write leaves the previous good file intact instead of a truncated one. Deterministic in, deterministic out.

**9. Validate as you clean — and log every drop.** Cleaning silently discards rows (nulls, dedup, unparseable dates); a silent drop is how a 5% coverage hole hides for a month. After each table, assert the cheap invariants — one row per key, dates within a sane range, null rates in the expected band, row counts in the right ballpark — and log a count for everything you remove. Fail loud on a surprise (an unexpected new category, a schema that grew a column). A cleaning run that "succeeds" with a quietly failed table is worse than one that stops.

**10. Prices come pre-adjusted — know which series you need.** Sharadar's `closeadj` is adjusted for splits *and* dividends (use it for total returns); `closeunadj` is the raw traded price (use it to reconstruct shares outstanding). Keep both. If you ever rebuild a split-sensitive series yourself — per-share dividends across a split, say — apply the cumulative split factors from ACTIONS consistently; a half-adjusted series is worse than either fully-adjusted or fully-raw.

## Pitfalls

- **The units trap.** Market cap in millions multiplied into a dollar-denominated dollar-volume is a 1,000× error that corrupts every cap cutoff and weight. Convert once, at ingestion.
- **Look-ahead through the wrong date.** Keying fundamentals on `calendardate` / `reportperiod` instead of `datekey` leaks 30–90 days of the future into every fundamental factor. It will *improve* your backtest — that's how you know it's wrong.
- **Ticker recycling.** Joining or chaining on the ticker string instead of `permaticker` splices unrelated companies together at reuse boundaries.
- **Rows with a null time key vanishing silently.** A partitioned writer that drops rows whose partition date is null will do it without telling you. Count them and investigate before you accept the loss.
- **Missing ≠ zero.** Treating a missing balance-sheet line as zero — or dropping the row — deletes a coherent cohort. The firms without that line item are usually a *type*, not noise.
- **Dedup that keeps an arbitrary row.** If your "keep one per key" doesn't actually preserve the order you intended, you keep a random duplicate — often a stale or partial one.
- **Metadata that isn't point-in-time.** Vendor sector/industry labels are usually a *current* snapshot stamped onto all of history. A company's sector today is not the sector it had in 2010. Know this; don't treat classification history as point-in-time unless you built it that way.
- **Trusting a clean-looking run.** No error is not the same as correct. The only real test of your cleaning is that the ESTU and the factors build on top of it and their numbers are sane.

## What "done" looks like

Cleaning is done when a later notebook can open `data/cleaned/` and *assume* the six properties without re-checking. Concretely, for each table you consume:

- ✅ Columns are correctly typed; dates are `Date`; keys are stable integers.
- ✅ The point-in-time column is identified, is the *availability* date, and the table is partitioned/keyed on it.
- ✅ Units are normalized and documented (market cap in dollars, not millions).
- ✅ Exactly one row per natural key, with a deliberate tiebreaker.
- ✅ Delisted names are retained; nothing was filtered for investability (that is the ESTU's job).
- ✅ A read-back spot-check passes — row counts, null rates, key uniqueness, and date ranges all in the expected band — and every dropped row was counted and explained.
- ✅ The result is partitioned, zstd-compressed parquet under `data/cleaned/`, and downstream steps read *only* from there.

**The real acceptance test is downstream.** If step 01 builds a sane ESTU and the first few factors produce sensible distributions and stable month-over-month rankings on top of your cleaned layer, the cleaning held. If a factor comes out with a flipped sign or a wrong order of magnitude, suspect the cleaning first — it is the correlated error source.

**Pro tip.** Keep the cleaning code boring and readable, not clever. It is the one part of the pipeline that everything else depends on and the one part you will rarely revisit — so write it for the version of you who comes back in six months to find out why one factor's coverage dipped in 2014.